# Sensor dark-frame calibration

Dark / read-noise characterisation of a camera through the ImSwitch REST API.
Written against a ToupTek camera but only uses generic endpoints
(`SettingsController` + `ReadNoiseCalibrationController`), so it works for any
detector ImSwitch exposes.

**Before you start: cover the sensor completely — put the cap on, kill the room
light. Everything below assumes true darkness (step 5 checks it).**

Steps:

1. cover the sensor (manual)
2. maximum gain, stable cooling, offset high enough that nothing clips at 0,
   all on-camera auto-corrections off
3. 20 dark frames at the minimum exposure
4. 10 dark frames at 100 ms
5. 10 dark frames at 10 s
6. 10 dark frames at 1000 s

Each stack lands in its own server-side session folder as a 16-bit TIFF stack
plus a `session.json` holding the exposure/gain it was taken with.

⚠️ The default plan runs for **~3 hours** (the 1000 s stack alone is 10 × 1000 s).
Trim `EXPOSURE_PLAN` while testing.

In [2]:
# pip install imswitchclient tifffile matplotlib
import json, time
import numpy as np
import imswitchclient.ImSwitchClient as imc

# ImSwitch API endpoint: "http://localhost:8001/imswitch/api" for a local
# instance, "http://192.168.178.76/imswitch/api" for a Pi behind Caddy (port 80).
# None uses $IMSWITCH_API_URL, which ImSwitch sets for notebooks it serves itself.
IMSWITCH_URL = "http://192.168.178.31:8001/imswitch/api"
DETECTOR = None        # None -> current detector

# (label, exposure in ms, number of frames). None = "camera minimum".
EXPOSURE_PLAN = [
    ("min",     None,       20),
    ("100ms",   100,        10),
    #("10s",     10_000,     10),
    #("1000s",   1_000_000,  10),
]

TARGET_TEMPERATURE_C = None   # e.g. -10 for a TEC camera, None = leave cooling alone
TEMPERATURE_TOLERANCE_C = 0.5
TEMPERATURE_TIMEOUT_S = 900

BLACKLEVEL = None      # None = keep current; raise it if the darkest pixels clip at 0
SESSION_PREFIX = "darkcal"

client = imc.ImSwitchClient(IMSWITCH_URL)
print("Connected to:", client.base_uri)
settings = client.settingsManager
readnoise = client.readnoiseManager

print("detectors:", settings.getDetectorNames())
DETECTOR = DETECTOR or readnoise.getStatus()["currentDetector"]
print("using:", DETECTOR)

Connected to: http://192.168.178.31:8001/imswitch/api
detectors: ['WidefieldCamera']
using: WidefieldCamera


## 2. Query the camera state as JSON

`getDetectorParameterTree` is the full, well-defined state: hardware info plus
every parameter with its type, editability and hardware limits. We keep a copy
to restore at the end.

In [3]:
def flat_params(tree):
    """{name: info} over all parameter groups of getDetectorParameterTree()."""
    return {p["name"]: p for g in tree.get("groups", []) for p in g["parameters"]}

tree0 = settings.getDetectorParameterTree(DETECTOR)
params0 = flat_params(tree0)
ORIGINAL_STATE = {n: p["value"] for n, p in params0.items() if p.get("editable")}

print(json.dumps({k: v for k, v in tree0.items() if k != "groups"}, indent=2)[:1200])
for name, p in params0.items():
    limits = "" if p.get("min") is None else f"min={p['min']} max={p['max']}"
    lock = "" if p.get("editable") else "(read-only) "
    print(f"{name:22s} = {str(p['value']):>12s}  [{p.get('type')}] {lock}"
          f"{p.get('options') or ''}{limits}")


{
  "model": "CameraToupcam",
  "isMock": false,
  "isConnected": true,
  "isRGB": false,
  "sensorWidth": 2992,
  "sensorHeight": 3000,
  "currentWidth": 2992,
  "currentHeight": 3000,
  "pixelSizeUm": [
    1,
    1.3,
    1.3
  ],
  "binning": 1,
  "supportedBinnings": [
    1,
    2,
    3,
    4
  ],
  "frameStart": [
    0,
    0
  ],
  "croppable": true,
  "forAcquisition": true,
  "forFocusLock": false,
  "cameraType": "Toupcam",
  "parameterRanges": {
    "exposure": {
      "min": 0.1,
      "max": 3600000.0,
      "units": "ms"
    },
    "gain": {
      "min": 0.0,
      "max": 23.0,
      "units": "arb.u."
    },
    "blacklevel": {
      "min": 0,
      "max": 1984,
      "units": "ADU"
    }
  },
  "isLongExposure": false,
  "isAcquiring": false,
  "isAdjustingParameters": false,
  "hardwareParameters": {
    "model_name": "MTR3CMOS09000KMA(USB2.0)",
    "serial_number": "ZP260305260120247",
    "firmware_version": "4.6.7.20221118",
    "hardware_version": "4.0",
    "se

## 3. Put the camera into a defined dark-calibration state

Every setting goes through `setDetectorParameterValue`, which casts the value to
the parameter's declared type and answers with the camera's own view of the
result — so a clamped or rejected value shows up immediately in the read-back.

On-camera corrections are switched off generically: any editable parameter whose
name looks like an auto-correction is set to its "off"/"manual" value. On a mono
ToupTek camera the SDK is already in RAW mode (no ISP: no denoise, sharpening,
white balance or gamma), so this is mostly a no-op there, and it does the right
thing on cameras that do expose those knobs.

In [4]:
CORRECTION_HINTS = ("auto", "awb", "denoise", "noise_reduction", "sharpen", "gamma",
                    "hdr", "flat_field", "flat_fielding", "defect", "contrast",
                    "saturation", "brightness_correction")

def set_param(name, value):
    """Set one parameter, return the value the camera reports back."""
    tree = settings.setDetectorParameterValue(name, value, DETECTOR)
    if "error" in tree:
        print(f"  ! {name}: {tree['error']}")
    return flat_params(tree).get(name, {}).get("value")

def apply_state(wanted):
    for name, value in wanted.items():
        if name not in params0:
            continue
        got = set_param(name, value)
        flag = "" if got == value else "  <-- clamped/rejected by hardware"
        print(f"  {name:22s} -> {got}{flag}")

wanted = {}

# manual exposure + no auto corrections
for name, p in params0.items():
    if not p.get("editable"):
        continue
    if not any(h in name.lower() for h in CORRECTION_HINTS):
        continue
    if p.get("type") == "boolean":
        wanted[name] = False
    elif p.get("type") == "list" and p.get("options"):
        off = next((o for o in p["options"] if str(o).lower() in ("manual", "off", "none", "false")), None)
        if off is not None:
            wanted[name] = off

# maximum gain
if params0.get("gain", {}).get("max") is not None:
    wanted["gain"] = params0["gain"]["max"]

# offset / black level
if BLACKLEVEL is not None:
    wanted["blacklevel"] = BLACKLEVEL

# full bit depth, no binning, free-running
if "pixel_format" in params0 and "mono16" in (params0["pixel_format"].get("options") or []):
    wanted["pixel_format"] = "mono16"
if TARGET_TEMPERATURE_C is not None and "target_temperature" in params0:
    wanted["target_temperature"] = TARGET_TEMPERATURE_C

apply_state(wanted)

if tree0.get("binning") != 1 and 1 in (tree0.get("supportedBinnings") or []):
    print("binning ->", settings.setDetectorBinning(DETECTOR, 1))

print()
print("state now:", json.dumps(readnoise.getDetectorSettings(DETECTOR)["settings"], indent=2))

  flat_fielding          -> False
  gain                   -> 23.0
  pixel_format           -> mono16

state now: {
  "exposure": 100.0,
  "gain": 23.0,
  "blacklevel": 0
}


## 4. Wait for the cooling to settle

Skipped when the camera has no `temperature` readout or `TARGET_TEMPERATURE_C`
is `None`.

In [5]:
def wait_for_temperature(target, tolerance=TEMPERATURE_TOLERANCE_C, timeout=TEMPERATURE_TIMEOUT_S):
    if target is None or "temperature" not in flat_params(settings.getDetectorParameterTree(DETECTOR)):
        print("no temperature readout / no target - skipping")
        return None
    t0 = time.time()
    while time.time() - t0 < timeout:
        temp = flat_params(settings.getDetectorParameterTree(DETECTOR))["temperature"]["value"]
        print(f"  {time.time()-t0:6.0f} s   {temp} C", end="\r")
        if temp is not None and abs(float(temp) - target) <= tolerance:
            print(f"\nstable at {temp} C")
            return temp
        time.sleep(10)
    print("\n! temperature did not stabilise within the timeout - continuing anyway")
    return temp

wait_for_temperature(TARGET_TEMPERATURE_C)

no temperature readout / no target - skipping


## 5. Confirm darkness and that nothing clips

`getHistogram` grabs a frame and reports min/max/mean/std plus the clipped
fraction. Two things to check before acquiring:

* **dark** — the mean should sit just above the offset, not in the thousands.
* **no clipping at 0** — if a large fraction of pixels sits exactly at the
  minimum, the offset is too low and the read-noise distribution is cut off.
  Raise `BLACKLEVEL` in the config cell and re-run step 3.

In [6]:
readnoise.setIllumination("off")   # lasers + LED matrices off, previous state remembered

# getHistogram reads the newest streamed frame, so use a short, defined exposure
set_param("exposure", 100)
time.sleep(1.0)

h = readnoise.getHistogram(DETECTOR, num_bins=128)
print(json.dumps({k: v for k, v in h.items() if k not in ("counts", "binEdges")}, indent=2))

counts, edges = np.asarray(h["counts"]), np.asarray(h["binEdges"])
at_zero = counts[0] / counts.sum()
print(f"\nfraction in the lowest bin: {at_zero:.3%}")
if h["min"] == 0 and at_zero > 0.01:
    print("!! clipping at 0 - raise BLACKLEVEL and re-run step 3")
if h.get("saturationFraction", 0) > 0:
    print("!! saturated pixels - is the sensor really covered?")

{
  "status": "ok",
  "min": 0.0,
  "max": 3742.0,
  "mean": 2.0608807932263815,
  "std": 35.55859900664131,
  "width": 2992,
  "height": 3000,
  "saturationCap": null,
  "saturationFraction": 0.0,
  "clipFraction": 1.1140819964349376e-07
}

fraction in the lowest bin: 99.245%
!! clipping at 0 - raise BLACKLEVEL and re-run step 3


## 6. Acquire the dark stacks

One session per exposure, so the stacks never overwrite each other and each one
carries its own `session.json` (detector, exposure, gain, blacklevel, timestamp).

The acquisition itself runs in a background thread on the server; `acquireStack`
polls `getProgress` until it finishes. The server sizes its per-frame timeout
from the exposure, so long exposures are waited out instead of being filled with
duplicates of the last frame.

In [7]:
exposure_min_ms = params0["exposure"].get("min") or settings.getDetectorParameters().get("exposureMin")
print("camera exposure range [ms]:", exposure_min_ms, "...", params0["exposure"].get("max"))

results = []
for label, exposure_ms, n_frames in EXPOSURE_PLAN:
    wanted_ms = exposure_min_ms if exposure_ms is None else exposure_ms
    applied = set_param("exposure", wanted_ms)
    if applied is not None and abs(float(applied) - float(wanted_ms)) > 1e-6:
        print(f"[{label}] requested {wanted_ms} ms, camera applied {applied} ms")

    session = readnoise.startSession(name=f"{SESSION_PREFIX}_{label}",
                                     detector_name=DETECTOR, nDark=n_frames)["session"]
    eta = n_frames * float(applied or wanted_ms) / 1000.0
    print(f"[{label}] {n_frames} frames @ {applied} ms -> {session['sessionId']} (~{eta/60:.1f} min)")

    progress = readnoise.acquireStack("dark", count=n_frames, poll=max(2.0, eta / 100),
                                      on_progress=lambda p: print("   ", p.get("message"), end="\r"))
    print("\n   ", progress.get("message"))

    results.append({"label": label, "exposureMs": applied, "frames": n_frames,
                    "sessionId": session["sessionId"],
                    "relPath": session["dataRelPath"] + "/dark/dark_stack.tif"})

readnoise.setIllumination("restore")
print(json.dumps(results, indent=2))

camera exposure range [ms]: 0.1 ... 3600000.0
[min] 20 frames @ 0.1 ms -> 20260913_131214_darkcal_min (~0.0 min)
    Saved 20 dark framesframes
    Saved 20 dark frames
s] 10 frames @ 100.0 ms -> 20260913_131228_darkcal_100ms (~0.0 min)
    Saved 10 dark framesrames
    Saved 10 dark frames
[
  {
    "label": "min",
    "exposureMs": 0.1,
    "frames": 20,
    "sessionId": "20260913_131214_darkcal_min",
    "relPath": "recordings/readnoise_calibration/20260913_131214_darkcal_min/dark/dark_stack.tif"
  },
  {
    "label": "100ms",
    "exposureMs": 100.0,
    "frames": 10,
    "sessionId": "20260913_131228_darkcal_100ms",
    "relPath": "recordings/readnoise_calibration/20260913_131228_darkcal_100ms/dark/dark_stack.tif"
  }
]


## 7. Fetch the stacks and summarise

The `recordings` tree is served statically, so the TIFF stacks download straight
over HTTP. Mean vs. exposure is the dark-current slope (ADU/s); the std at the
minimum exposure is the read noise in ADU.

In [8]:
import io, requests, tifffile

summary = []
for r in results:
    url = readnoise.dataUrl(r["relPath"])
    stack = tifffile.imread(io.BytesIO(requests.get(url, verify=False).content))
    per_frame_mean = stack.reshape(stack.shape[0], -1).mean(axis=1)
    summary.append({**r, "shape": stack.shape, "dtype": str(stack.dtype),
                    "mean": float(stack.mean()), "std": float(stack.std()),
                    "temporalStd": float(stack.std(axis=0).mean()),
                    "frameToFrameSpread": float(per_frame_mean.std()),
                    "min": int(stack.min()), "max": int(stack.max())})
    print(f"{r['label']:>8s}  {r['exposureMs']:>12} ms  mean={summary[-1]['mean']:9.2f}  "
          f"temporal sigma={summary[-1]['temporalStd']:7.2f}  min={summary[-1]['min']}  max={summary[-1]['max']}")

# dark current from the mean vs. exposure slope
x = np.array([s["exposureMs"] for s in summary], dtype=float) / 1000.0
y = np.array([s["mean"] for s in summary], dtype=float)
if len(x) > 1:
    slope, offset = np.polyfit(x, y, 1)
    print(f"\ndark current ~ {slope:.4f} ADU/s, offset ~ {offset:.2f} ADU")

json.dump(summary, open("dark_calibration_summary.json", "w"), indent=2)

     min           0.1 ms  mean=     1.95  temporal sigma=   5.45  min=0  max=4178
   100ms         100.0 ms  mean=     1.81  temporal sigma=   4.13  min=0  max=7576

dark current ~ -1.3640 ADU/s, offset ~ 1.95 ADU


## 8. Restore the camera

Puts every editable parameter back to the value it had before this notebook ran.

In [ ]:
apply_state(ORIGINAL_STATE)
readnoise.setIllumination("restore")